<a href="https://colab.research.google.com/github/jaywong8/Detection-of-Atrial-Fibrillation-Termination/blob/src/(Unfinished)_MultiRecord_Pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Icentia11k AFib Termination — Multi-record pilot

**Purpose:** download a limited set of ECG records, inspect rhythm annotations, identify explicitly annotated AFib-to-normal-sinus-rhythm (NSR) transitions, and build an initial feature table for logistic regression.

**Important:** This is a *pilot / label-validation notebook*, not yet a validated clinical prediction pipeline. A `)` annotation alone is not evidence of NSR. Positive outcomes require an explicit `(N` rhythm annotation after AFib. Ambiguous endings, other rhythms, and insufficient follow-up are excluded from the first-pass labels.

Dataset: [Icentia11k on PhysioNet](https://physionet.org/content/icentia11k-continuous-ecg/1.0/)

## 1. Install libraries

In [ ]:
!pip -q install wfdb pandas numpy scipy scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 2.9 MB/s eta 0:00:00


## 2. Configure the pilot

Start with 20–30 distinct patient IDs. The list below is just a *seed list* to test the workflow; it does not guarantee that these patients contain usable AFib-to-NSR transitions. Add more IDs after reviewing the output.

Each segment is approximately 70 minutes. Start with segment `00` only; later add segment IDs for patients with useful annotations.

In [ ]:
from pathlib import Path
import urllib.request
import time
import warnings
import numpy as np
import pandas as pd
import wfdb
import matplotlib.pyplot as plt

BASE = "https://physionet.org/files/icentia11k-continuous-ecg/1.0"
DATA_DIR = Path("/content/icentia_pilot")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Initial pilot: 30 distinct patient IDs. Expand after reviewing usable outcomes.
PATIENT_IDS = list(range(20, 50))  # 30 distinct patient IDs; missing records are logged and skipped
SEGMENT_IDS = [0]  # begin with segment 00 only

FS_EXPECTED = 250
WINDOW_SEC = 30        # ECG history available to the model
HORIZON_SEC = 60       # predict NSR transition within the next 60 seconds
STEP_SEC = 10          # prediction opportunities every 10 seconds (pilot)
MIN_AFIB_SEC = 15      # avoid very short annotated AFib regions in this pilot

print("Requested unique patients:", len(PATIENT_IDS))
print("Planned records:", len(PATIENT_IDS) * len(SEGMENT_IDS))

Requested unique patients: 30
Planned records: 30


## 3. Build valid PhysioNet URLs and download only selected files

The directory rule is `p{patient_id:05d}` and the grouping directory is `p0{first digit of patient_id}` for this dataset (for example, patient 60 → `p00/p00060/`). Segment IDs are formatted with two digits. Download the `.hea`, `.dat`, and `.atr` files for each selected record only.

If a file is missing, skip that record and log the error rather than assuming all IDs/segments exist.

In [ ]:
def record_stem(patient_id, segment_id):
    patient = f"p{patient_id:05d}"
    group = patient[:3]
    segment = f"{patient}_s{segment_id:02d}"
    return patient, group, segment

def download_record(patient_id, segment_id, retries=2):
    patient, group, segment = record_stem(patient_id, segment_id)
    local_dir = DATA_DIR / patient
    local_dir.mkdir(parents=True, exist_ok=True)
    local_stem = local_dir / segment

    url_dir = f"{BASE}/{group}/{patient}"
    for ext in ["hea", "dat", "atr"]:
        dest = Path(f"{local_stem}.{ext}")
        if dest.exists() and dest.stat().st_size > 0:
            continue
        url = f"{url_dir}/{segment}.{ext}"
        last_error = None
        for attempt in range(retries + 1):
            try:
                urllib.request.urlretrieve(url, dest)
                if dest.stat().st_size == 0:
                    raise IOError("Downloaded empty file")
                last_error = None
                break
            except Exception as e:
                last_error = e
                if dest.exists():
                    dest.unlink()
                time.sleep(0.5 * (attempt + 1))
        if last_error is not None:
            raise last_error
    return str(local_stem)

download_log = []
record_paths = []
for pid in PATIENT_IDS:
    for sid in SEGMENT_IDS:
        try:
            stem = download_record(pid, sid)
            record_paths.append((pid, sid, stem))
            download_log.append({"patient_id": pid, "segment_id": sid, "status": "downloaded", "path": stem})
            print("OK:", stem)
        except Exception as e:
            download_log.append({"patient_id": pid, "segment_id": sid, "status": "failed", "error": str(e)})
            print("SKIP:", pid, sid, repr(e))

pd.DataFrame(download_log)

OK: /content/icentia_pilot/p00020/p00020_s00
OK: /content/icentia_pilot/p00021/p00021_s00
OK: /content/icentia_pilot/p00022/p00022_s00
OK: /content/icentia_pilot/p00023/p00023_s00
OK: /content/icentia_pilot/p00024/p00024_s00
OK: /content/icentia_pilot/p00025/p00025_s00
OK: /content/icentia_pilot/p00026/p00026_s00
OK: /content/icentia_pilot/p00027/p00027_s00
OK: /content/icentia_pilot/p00028/p00028_s00
OK: /content/icentia_pilot/p00029/p00029_s00
OK: /content/icentia_pilot/p00030/p00030_s00
OK: /content/icentia_pilot/p00031/p00031_s00
OK: /content/icentia_pilot/p00032/p00032_s00
OK: /content/icentia_pilot/p00033/p00033_s00
OK: /content/icentia_pilot/p00034/p00034_s00
OK: /content/icentia_pilot/p00035/p00035_s00
OK: /content/icentia_pilot/p00036/p00036_s00
OK: /content/icentia_pilot/p00037/p00037_s00
OK: /content/icentia_pilot/p00038/p00038_s00
OK: /content/icentia_pilot/p00039/p00039_s00
OK: /content/icentia_pilot/p00040/p00040_s00
OK: /content/icentia_pilot/p00041/p00041_s00
OK: /conte

,patient_id,segment_id,status,path
0,20,0,downloaded,/content/icentia_pilot/p00020/p00020_s00
1,21,0,downloaded,/content/icentia_pilot/p00021/p00021_s00
2,22,0,downloaded,/content/icentia_pilot/p00022/p00022_s00
3,23,0,downloaded,/content/icentia_pilot/p00023/p00023_s00
4,24,0,downloaded,/content/icentia_pilot/p00024/p00024_s00
5,25,0,downloaded,/content/icentia_pilot/p00025/p00025_s00
6,26,0,downloaded,/content/icentia_pilot/p00026/p00026_s00
7,27,0,downloaded,/content/icentia_pilot/p00027/p00027_s00
8,28,0,downloaded,/content/icentia_pilot/p00028/p00028_s00
9,29,0,downloaded,/content/icentia_pilot/p00029/p00029_s00


## 4. Inspect the annotation vocabulary

WFDB `aux_note` includes rhythm labels such as `(AFIB` and sometimes `(N` for normal sinus rhythm. The beat symbol `N` is different from the rhythm annotation `(N`. The `)` marker is not treated as proof of NSR.

This cell summarizes the actual annotations found in the selected records before attempting labels.

In [ ]:
def read_annotations(stem):
    ann = wfdb.rdann(stem, "atr")
    events = []
    aux = getattr(ann, "aux_note", None)
    if aux is None:
        aux = [None] * len(ann.sample)
    for sample, note in zip(ann.sample, aux):
        if note is None:
            continue
        note = str(note).strip()
        if not note or note.lower() == "none":
            continue
        events.append((int(sample), note))
    events.sort(key=lambda x: x[0])
    return ann, events

annotation_summaries = []
annotation_events = {}
for pid, sid, stem in record_paths:
    try:
        ann, events = read_annotations(stem)
        annotation_events[(pid, sid)] = events
        rhythm_notes = [note for _, note in events if note.startswith("(") or note == ")"]
        counts = pd.Series(rhythm_notes).value_counts().to_dict()
        annotation_summaries.append({
            "patient_id": pid, "segment_id": sid,
            "n_annotations": len(ann.sample),
            "AFIB_starts": sum(n.upper().startswith("(AFIB") for n in rhythm_notes),
            "NSR_starts_(N": sum(n == "(N" for n in rhythm_notes),
            "AFL_starts": sum(n.upper().startswith("(AFL") for n in rhythm_notes),
            "close_markers_')'": sum(n == ")" for n in rhythm_notes),
            "rhythm_annotation_counts": counts
        })
    except Exception as e:
        annotation_summaries.append({"patient_id": pid, "segment_id": sid, "error": repr(e)})

annotation_summary_df = pd.DataFrame(annotation_summaries)
display(annotation_summary_df)

,patient_id,segment_id,n_annotations,AFIB_starts,NSR_starts_(N,AFL_starts,close_markers_')',rhythm_annotation_counts
0,20,0,5767,0,74,0,74,"{'(N': 74, ')': 74}"
1,21,0,6080,0,71,0,71,"{'(N': 71, ')': 71}"
2,22,0,7718,0,58,0,58,"{'(N': 58, ')': 58}"
3,23,0,6031,0,46,0,46,"{'(N': 46, ')': 46}"
4,24,0,6217,0,26,0,26,"{'(N': 26, ')': 26}"
5,25,0,6834,0,86,0,86,"{'(N': 86, ')': 86}"
6,26,0,5420,0,10,0,10,"{'(N': 10, ')': 10}"
7,27,0,5912,0,52,0,52,"{'(N': 52, ')': 52}"
8,28,0,5686,0,76,0,76,"{'(N': 76, ')': 76}"
9,29,0,5674,0,33,0,33,"{'(N': 33, ')': 33}"


## 5. Parse AFib regions conservatively

This first-pass parser closes an AFib region at the next rhythm boundary (`(AFIB`, `(N`, `(AFL`, or `)`). Crucially, it marks the *end type* and only considers a transition to NSR confirmed when the next boundary is explicitly `(N`.

Because annotation conventions and edge cases must be validated, inspect several parsed regions against the raw annotation sequence before using labels for training.

In [ ]:
RHYTHM_STARTS = ("(AFIB", "(N", "(AFL")

def parse_afib_regions(events):
    regions = []
    for i, (start, note) in enumerate(events):
        if not note.upper().startswith("(AFIB"):
            continue

        end_sample = None
        end_note = None
        for sample2, note2 in events[i + 1:]:
            if sample2 <= start:
                continue
            if note2 == ")" or note2.startswith("("):
                end_sample, end_note = sample2, note2
                break

        if end_sample is None:
            regions.append({
                "start": start, "end": None, "end_note": None,
                "transition": "unknown_open_ended"
            })
            continue

        if end_note == "(N":
            transition = "explicit_NSR"
        elif end_note == ")":
            transition = "unconfirmed_close"
        elif end_note.upper().startswith("(AFIB"):
            transition = "AFIB_reannotation_or_boundary"
        else:
            transition = "other_rhythm"

        regions.append({
            "start": start, "end": end_sample, "end_note": end_note,
            "transition": transition
        })
    return regions

all_regions = []
for pid, sid, stem in record_paths:
    events = annotation_events.get((pid, sid), [])
    try:
        record = wfdb.rdrecord(stem, physical=False)
        fs = float(record.fs)
        n_samples = int(record.sig_len)
        for r in parse_afib_regions(events):
            all_regions.append({
                "patient_id": pid, "segment_id": sid,
                "record_stem": stem, "fs": fs, "n_samples": n_samples,
                **r,
                "start_sec": r["start"] / fs,
                "end_sec": (r["end"] / fs) if r["end"] is not None else np.nan,
                "duration_sec": ((r["end"] - r["start"]) / fs) if r["end"] is not None else np.nan
            })
    except Exception as e:
        print("Could not parse", pid, sid, repr(e))

regions_df = pd.DataFrame(all_regions)
display(regions_df.head(30))
if len(regions_df):
    display(regions_df.groupby(["transition"]).size().rename("count").to_frame())

""


## 6. Define prediction windows and outcomes

**Pilot target:** while the annotation indicates AFib, use the previous 30 seconds of ECG to predict whether an explicit `(N` annotation begins within the next 60 seconds.

For this conservative pilot:
- Positive: the current point is inside a parsed AFib region and an explicit `(N` boundary occurs within the next 60 seconds.
- Negative: AFib continues for the full 60-second horizon, with no competing rhythm boundary.
- Exclude: a `)` end marker without explicit NSR, another rhythm, too little prior ECG, or insufficient follow-up.
- Features use only ECG from `[t - 30 s, t)`. Labels look forward from `t`; future samples must never enter the features.

This is a first-pass label definition. It still requires validation against the dataset annotation conventions and does not prove the transition was spontaneous or untreated.

In [ ]:
def extract_features(x, fs):
    # x is physical ECG signal from the input window.
    x = np.asarray(x, dtype=float).reshape(-1)
    x = x[np.isfinite(x)]
    if len(x) < max(10, int(fs * 5)):
        return None
    # Signal-only quality summaries; not a full signal-quality classifier.
    q = np.quantile(x, [0.01, 0.25, 0.5, 0.75, 0.99])
    return {
        "ecg_mean": float(np.mean(x)),
        "ecg_std": float(np.std(x)),
        "ecg_min": float(np.min(x)),
        "ecg_max": float(np.max(x)),
        "ecg_p01": float(q[0]),
        "ecg_median": float(q[2]),
        "ecg_p99": float(q[4]),
        "ecg_iqr": float(q[3] - q[1]),
        "ecg_rms": float(np.sqrt(np.mean(x**2))),
        "ecg_diff_std": float(np.std(np.diff(x))) if len(x) > 1 else np.nan,
        "finite_fraction": float(np.mean(np.isfinite(np.asarray(x))))
    }

def load_record_physical(stem):
    rec = wfdb.rdrecord(stem, physical=True)
    sig = np.asarray(rec.p_signal[:, 0], dtype=float)
    return sig, float(rec.fs)

# Build a pilot dataset.
# To avoid reading every waveform into RAM at once, process one record at a time.
rows = []
exclusion_counts = {}

def excluded(reason):
    exclusion_counts[reason] = exclusion_counts.get(reason, 0) + 1

for pid, sid, stem in record_paths:
    try:
        signal, fs = load_record_physical(stem)
        events = annotation_events.get((pid, sid), [])
        regions = parse_afib_regions(events)
        win = int(WINDOW_SEC * fs)
        horizon = int(HORIZON_SEC * fs)
        step = max(1, int(STEP_SEC * fs))

        # For each AFib region, place prediction times at fixed intervals.
        for r in regions:
            start = int(r["start"])
            end = r["end"]
            if end is None or (end - start) / fs < MIN_AFIB_SEC:
                excluded("short_or_open_afib_region")
                continue

            # Start after enough ECG history exists and ensure t remains within AFib.
            t_min = max(start, win)
            for t in range(t_min, end, step):
                if t - win < 0 or t >= len(signal):
                    excluded("insufficient_input")
                    continue

                # Label only when the explicit NSR boundary lies in the forward horizon.
                # Positive is possible only if this AFib region ends in explicit (N).
                if r["transition"] == "explicit_NSR" and end > t and end <= t + horizon:
                    y = 1
                elif (end >= t + horizon and
                      r["transition"] not in ("other_rhythm", "unconfirmed_close",
                                               "unknown_open_ended", "AFIB_reannotation_or_boundary")):
                    y = 0
                else:
                    excluded("ambiguous_or_outside_horizon")
                    continue

                x = signal[t-win:t]
                feat = extract_features(x, fs)
                if feat is None:
                    excluded("invalid_features")
                    continue

                rows.append({
                    **feat,
                    "patient_id": pid,
                    "segment_id": sid,
                    "record_stem": stem,
                    "prediction_sample": t,
                    "prediction_time_sec": t / fs,
                    "afib_start_sample": start,
                    "afib_end_sample": end,
                    "label_nsr_within_horizon": int(y),
                    "window_sec": WINDOW_SEC,
                    "horizon_sec": HORIZON_SEC
                })
    except Exception as e:
        print("Record processing failed:", pid, sid, repr(e))

dataset = pd.DataFrame(rows)
print("Rows:", len(dataset))
print("Patients with rows:", dataset["patient_id"].nunique() if len(dataset) else 0)
print("Label counts:")
display(dataset["label_nsr_within_horizon"].value_counts(dropna=False).rename("n").to_frame() if len(dataset) else pd.DataFrame())
print("Exclusions:", exclusion_counts)
display(dataset.head())

## 7. Check whether the pilot is viable

Do not train a model if the table has only one class, very few patients, or no positive outcomes. Expand the patient list / segment IDs and rerun. Keep a record of which patients were scanned and why records were excluded.

In [ ]:
if len(dataset):
    print("Rows per patient:")
    display(dataset.groupby("patient_id").size().rename("windows").to_frame())
    print("Positive rate:", dataset["label_nsr_within_horizon"].mean())
    print("Unique patients per label:")
    display(dataset.groupby("label_nsr_within_horizon")["patient_id"].nunique().rename("patients").to_frame())
else:
    print("No model-ready windows yet. Expand the pilot or review label/parser logic.")

## 8. Patient-level train / validation / test split

Never randomly split windows: windows from the same person can be highly correlated. Split by patient first, then assign all windows from each patient to exactly one set. With a small pilot, a split may fail to contain both classes; that is a sign to collect more eligible patients, not to split by window.

The held-out test set should not be used for feature selection, threshold tuning, or model choices.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

if len(dataset) and dataset["patient_id"].nunique() >= 5 and dataset["label_nsr_within_horizon"].nunique() == 2:
    # First reserve 20% of patients for test.
    gss1 = GroupShuffleSplit(n_splits=50, test_size=0.20, random_state=42)
    split1 = None
    for trainval_idx, test_idx in gss1.split(dataset, dataset["label_nsr_within_horizon"], groups=dataset["patient_id"]):
        if dataset.iloc[trainval_idx]["label_nsr_within_horizon"].nunique() == 2:
            split1 = (trainval_idx, test_idx)
            break

    if split1 is None:
        print("Could not create a training split with both classes. More patients/outcomes needed.")
    else:
        trainval_idx, test_idx = split1
        trainval = dataset.iloc[trainval_idx].copy()
        test = dataset.iloc[test_idx].copy()

        # Reserve 20% of the remaining patients (~16% overall) for validation.
        gss2 = GroupShuffleSplit(n_splits=50, test_size=0.20, random_state=43)
        split2 = None
        for tr_idx, val_idx in gss2.split(trainval, trainval["label_nsr_within_horizon"], groups=trainval["patient_id"]):
            if trainval.iloc[tr_idx]["label_nsr_within_horizon"].nunique() == 2:
                split2 = (tr_idx, val_idx)
                break

        if split2 is None:
            print("Could not create a training split with both classes. More patients/outcomes needed.")
        else:
            tr_idx, val_idx = split2
            train = trainval.iloc[tr_idx].copy()
            val = trainval.iloc[val_idx].copy()

            print("Patient counts:", train.patient_id.nunique(), val.patient_id.nunique(), test.patient_id.nunique())
            print("Patient overlap:", set(train.patient_id) & set(val.patient_id),
                  set(train.patient_id) & set(test.patient_id),
                  set(val.patient_id) & set(test.patient_id))
            display(pd.DataFrame({
                "split": ["train", "validation", "test"],
                "rows": [len(train), len(val), len(test)],
                "patients": [train.patient_id.nunique(), val.patient_id.nunique(), test.patient_id.nunique()],
                "positive_rate": [train.label_nsr_within_horizon.mean(),
                                  val.label_nsr_within_horizon.mean(),
                                  test.label_nsr_within_horizon.mean()]
            }))
else:
    print("Need both labels and at least 5 unique patients before attempting this pilot split.")

## 9. Train a baseline logistic regression (only after label review)

The features in this initial notebook are waveform summaries, not the final planned RR-interval feature set. The beat annotation handling and RR extraction should be validated separately before treating RR features as reliable.

This section fits a minimal baseline only if the split exists and each set has both classes. Fit imputation/scaling on training data only. Select any probability threshold using validation data only. Evaluate test data once after the pipeline is finalized.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

FEATURES = [
    "ecg_mean", "ecg_std", "ecg_min", "ecg_max",
    "ecg_p01", "ecg_median", "ecg_p99", "ecg_iqr",
    "ecg_rms", "ecg_diff_std", "finite_fraction"
]

if all(name in globals() for name in ["train", "val", "test"]) and train["label_nsr_within_horizon"].nunique() == 2:
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42))
    ])
    model.fit(train[FEATURES], train["label_nsr_within_horizon"])

    for split_name, split_df in [("train", train), ("validation", val), ("test", test)]:
        if split_df["label_nsr_within_horizon"].nunique() < 2:
            print(split_name, "has one class; AUROC undefined.")
            continue
        probs = model.predict_proba(split_df[FEATURES])[:, 1]
        print(split_name,
              "AUROC =", round(roc_auc_score(split_df["label_nsr_within_horizon"], probs), 3),
              "AUPRC =", round(average_precision_score(split_df["label_nsr_within_horizon"], probs), 3))
else:
    print("Skipped: first confirm enough patients and both outcome classes.")

## 10. Save the pilot table and audit logs

Save the derived table, annotation summary, and download log. These do not replace the original source files; they document the cohort and help reproduce the pilot.

In [ ]:
dataset.to_csv("/content/icentia_pilot_windows.csv", index=False)
annotation_summary_df.to_csv("/content/icentia_annotation_summary.csv", index=False)
pd.DataFrame(download_log).to_csv("/content/icentia_download_log.csv", index=False)
print("Saved CSVs in /content/")

## Interpretation and next steps

1. Verify the parser against several manually inspected examples, especially explicit `(N` transitions.
2. Expand to 20–30 unique patients, then measure the number of eligible patients, episodes, prediction windows, and positive outcomes.
3. Confirm labels and exclusion logic before model fitting.
4. Improve features with validated beat-to-beat intervals and signal-quality checks; do not assume all beat annotations are normal sinus beats.
5. Use patient-level splits, class-aware metrics (especially AUPRC), calibration, and patient-aware uncertainty estimates.
6. Treat this as research on annotated rhythm transitions. The dataset alone does not establish that termination was spontaneous or that treatment did not occur.